# TOPOTEX Dataset Inspector — object-level baseline

正式数据集人工验收：**4,590 complete objects（train 4,090 / unseen test 500）**。
Split 单位 = source object（GLB 资产）；同一 object 的全部 query 同属一个 split。

| on-disk | 语义 | 说明 |
|---|---|---|
| `uv_000` | **native** / full | 原生 GLB 参数化 |
| `uv_001` | **xatlas** / full | xatlas 重参数化 |
| `uv_test` | **blender_smart** / full | Blender Smart UV |
| `uv_002` | **partial** = native / connected_partial | **surface mask/query，不是 unwrap family** |

训练采样：full 0.80（三 layout 均匀）+ partial 0.20；泛化轴 = **unseen objects**。

In [ ]:
import json, os, sys
from pathlib import Path
p = Path.cwd()
PROJECT_ROOT = next(c for c in (p, *p.parents) if (c / "configs").exists())
sys.path.insert(0, str(PROJECT_ROOT))

# ------------------------------------------------ configuration
DATA = Path(os.environ.get("TOPOTEX_DATA", "/root/youjiaZhang/topotex_data"))
DATASET_ROOT = Path(os.environ.get("TOPOTEX_DATASET_ROOT", DATA / "dataset"))
SPLIT_PATH = Path(os.environ.get("TOPOTEX_SPLIT", DATA / "object_split.json"))
SUBSET = "train"            # "train" | "test"
SAMPLE_ID = None            # explicit id overrides random pick
RANDOM_SAMPLE = True
RANDOM_SEED = 20260727
MODE = "quick"              # "quick": 4+4 spot check | "full": 16+16

import numpy as np
import torch
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image
from safetensors.numpy import load_file

matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK SC",
                                          "Noto Sans CJK JP", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False

split = json.loads(SPLIT_PATH.read_text())
TRAIN, TEST = split["train"], split["val"]
GLB_SHA = split["source_glb_sha256"]
ids = [json.loads(l)["sample_id"] for l in open(DATASET_ROOT / "manifest.jsonl")]
SEM = {"uv_000": ("native", "full"), "uv_001": ("xatlas", "full"),
       "uv_002": ("partial(native)", "connected_partial"),
       "uv_test": ("blender_smart", "full")}
FULLS = ["uv_000", "uv_001", "uv_test"]
print(f"objects {len(ids)} | train {len(TRAIN)} | unseen test {len(TEST)}")
assert len(ids) == len(TRAIN) + len(TEST) == 4590

## Dataset Overview — 冻结产物与统计

In [ ]:
import hashlib
for name in ("dataset_manifest.jsonl", "object_split.json",
             "dataset_statistics.json", "source_resolution_audit.json"):
    h = hashlib.sha256((DATA / name).read_bytes()).hexdigest()
    print(f"{name:32s} sha256 {h[:16]}…")
print("source_manifest_current        ", open(DATA / "source_manifest_current.sha256").read().split()[0][:16] + "…")
audit = json.loads((DATA / "source_resolution_audit.json").read_text())
print("\nresolution audit:", {k: v for k, v in audit.items()
      if "resolution" in k and isinstance(v, (int, dict))})
st = json.loads((DATA / "dataset_statistics.json").read_text())
src = st["source"]
fig, axes = plt.subplots(1, 3, figsize=(14, 3.2))
for ax, key, ttl in ((axes[0], "face_count", "face count"),
                     (axes[1], "vertex_count", "vertex count")):
    h = src[key]["hist"]
    ax.bar(h["edges"][:-1], h["counts"], width=np.diff(h["edges"]), align="edge")
    ax.set_title(f"{ttl} (median {src[key]['median']})", fontsize=9)
STATNAME = {"canonical": "native", "alternative": "xatlas",
            "partial": "partial", "heldout": "blender_smart"}
for nm, o in st["uv_query_dataset"]["uv_occupancy"].items():
    h = o["hist"]
    centers = [(h["edges"][i] + h["edges"][i+1]) / 2 for i in range(len(h["counts"]))]
    axes[2].plot(centers, h["counts"], label=STATNAME.get(nm, nm), lw=1.2)
axes[2].legend(fontsize=7); axes[2].set_title("UV occupancy per query", fontsize=9)
plt.tight_layout(); plt.show()

### 连通分量与 island 分布（quick 模式抽样 256 个对象实算）

In [ ]:
from scipy import ndimage
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

rng = np.random.default_rng(RANDOM_SEED)
probe = [ids[i] for i in rng.choice(len(ids), size=256 if MODE == "quick" else 1024, replace=False)]
comp_counts, island_counts = [], {q: [] for q in FULLS}
for sid in probe:
    d = DATASET_ROOT / "samples" / sid
    F = load_file(str(d / "mesh.safetensors"))["faces"]
    edges = {}
    for tri in range(len(F)):
        a, b, c = sorted(F[tri])
        for e in ((a, b), (b, c), (a, c)):
            edges.setdefault(e, []).append(tri)
    r, cc = [], []
    for tris in edges.values():
        for i in range(len(tris) - 1):
            r.append(tris[i]); cc.append(tris[i + 1])
    if r:
        g = coo_matrix((np.ones(len(r)), (r, cc)), shape=(len(F), len(F)))
        n, _ = connected_components(g, directed=False)
    else:
        n = len(F)
    comp_counts.append(n)
    for q in FULLS:
        vm = load_file(str(d / "uv_queries" / q / "uv_address.safetensors"))["valid_mask"]
        _, ni = ndimage.label(vm)
        island_counts[q].append(ni)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
axes[0].hist(comp_counts, bins=40)
axes[0].set_title(f"mesh connected components (n={len(probe)}, median {int(np.median(comp_counts))})", fontsize=9)
for q in FULLS:
    axes[1].hist(island_counts[q], bins=40, alpha=0.5, label=SEM[q][0])
axes[1].legend(fontsize=8); axes[1].set_title("UV island count per layout", fontsize=9)
plt.tight_layout(); plt.show()

## Random Object Inspector — A. Source inputs / B. Metadata

In [ ]:
from topotex.data.mesh import CANONICAL_VIEWS, camera_matrices, rasterize_view

pool = TRAIN if SUBSET == "train" else TEST
rng = np.random.default_rng(RANDOM_SEED)
sid = SAMPLE_ID or (pool[int(rng.integers(len(pool)))] if RANDOM_SAMPLE else pool[0])
d = DATASET_ROOT / "samples" / sid
meta = json.loads((d / "meta.json").read_text())
mesh = load_file(str(d / "mesh.safetensors"))
V, F = mesh["vertices"].astype(np.float64), mesh["faces"].astype(np.int64)
mv = load_file(str(d / "mv.safetensors"))["images"]

def render_white(vi, res=384):
    _, az, el = CANONICAL_VIEWS[vi]
    gb = rasterize_view(V, F, camera_matrices(az, el, V.min(0), V.max(0)), res)
    tri = V[F]
    n = np.cross(tri[:, 1] - tri[:, 0], tri[:, 2] - tri[:, 0])
    n /= np.clip(np.linalg.norm(n, axis=1, keepdims=True), 1e-12, None)
    lam = np.abs(n @ (np.array([0.4, 0.6, 0.7]) / np.linalg.norm([0.4, 0.6, 0.7])))
    img = np.ones((res, res, 3)); m = gb["mask"]
    img[m] = (0.25 + 0.75 * lam[gb["face_id"][m]])[:, None] * np.array([0.86, 0.87, 0.9])
    return img

fig = plt.figure(figsize=(16, 3))
ax = fig.add_subplot(1, 8, 1)
ax.imshow(np.asarray(Image.open(d / "reference.png").convert("RGB")))
ax.set_title("reference (512²)", fontsize=9); ax.axis("off")
ax = fig.add_subplot(1, 8, 2)
if torch.cuda.is_available():
    ax.imshow(render_white(0))
ax.set_title("textureless mesh", fontsize=9); ax.axis("off")
for i in range(6):
    ax = fig.add_subplot(1, 8, i + 3)
    ax.imshow(mv[i].transpose(1, 2, 0))
    ax.set_title(CANONICAL_VIEWS[i][0], fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

edges = {}
for tri in range(len(F)):
    a, b, c = sorted(F[tri])
    for e in ((a, b), (b, c), (a, c)):
        edges.setdefault(e, []).append(tri)
import scipy.sparse as sp
from scipy.sparse.csgraph import connected_components as _cc
r = [t[i] for t in edges.values() for i in range(len(t) - 1)]
cc = [t[i + 1] for t in edges.values() for i in range(len(t) - 1)]
ncomp = _cc(sp.coo_matrix((np.ones(len(r)), (r, cc)), shape=(len(F), len(F))), directed=False)[0] if r else len(F)
print("sample_id / asset_id :", sid)
print("source GLB sha256    :", GLB_SHA[sid][:24] + "…")
print("split                :", "train" if sid in set(TRAIN) else "TEST (unseen)")
print("vertices / faces     :", len(V), "/", len(F), "| connected components:", ncomp)
print("resolutions          : reference 512² | MV 256² (UniTEX native 512) | native GT 256² | model/query 256²")
print("query builder commit :", meta.get("query_builder_commit"))

## C. 三个 full UV layouts（native / xatlas / blender_smart）

In [ ]:
from scipy import ndimage
fig, axes = plt.subplots(3, 5, figsize=(15, 9.6))
for r_i, q in enumerate(FULLS):
    ua = load_file(str(d / "uv_queries" / q / "uv_address.safetensors"))
    vm = ua["valid_mask"].astype(bool)
    fid = ua["face_id"].astype(np.float64); fid[fid < 0] = np.nan
    bary = ua["barycentric"].astype(np.float64).transpose(1, 2, 0); bary[~vm] = 0
    gt = np.asarray(Image.open(d / "uv_queries" / q / "gt_texture.png"))
    uvv, uvf = ua["uv_vertices"], ua["uv_faces"].astype(np.int64)
    _, nisl = ndimage.label(vm)
    axes[r_i][0].triplot(uvv[:, 0], 1 - uvv[:, 1], uvf, lw=0.15, color="tab:blue")
    axes[r_i][0].set_xlim(0, 1); axes[r_i][0].set_ylim(0, 1); axes[r_i][0].set_aspect(1)
    axes[r_i][0].set_ylabel(SEM[q][0], fontsize=11)
    axes[r_i][1].imshow(gt)
    axes[r_i][2].imshow(fid, cmap="nipy_spectral")
    axes[r_i][3].imshow(bary)
    axes[r_i][4].imshow(vm, cmap="gray")
    axes[r_i][4].set_xlabel(f"occ {vm.mean():.2f} | islands {nisl}", fontsize=9)
for c_i, t in enumerate(["UV layout", "GT texture", "face_id", "barycentric", "valid mask"]):
    axes[0][c_i].set_title(t, fontsize=10)
for a in axes.ravel():
    a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

## D. Partial query — **surface mask/query（不是 unwrap family）**
base layout = native；连通面子集重光栅化，GT 在子集内重烘焙、区外精确为 0。

In [ ]:
ua0 = load_file(str(d / "uv_queries/uv_000/uv_address.safetensors"))
uap = load_file(str(d / "uv_queries/uv_002/uv_address.safetensors"))
vm0, vmp = ua0["valid_mask"].astype(bool), uap["valid_mask"].astype(bool)
gtp = np.asarray(Image.open(d / "uv_queries/uv_002/gt_texture.png"))
sel_faces = np.unique(uap["face_id"][vmp]); sel_faces = sel_faces[sel_faces >= 0]
qm = {q["name"]: q for q in meta["uv_queries"]}
fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
axes[0].imshow(vm0, cmap="gray"); axes[0].set_title("base layout (native) valid", fontsize=9)
sel = np.zeros_like(vm0, dtype=float); sel[vmp] = 1; sel[vm0 & ~vmp] = 0.35
axes[1].imshow(sel, cmap="viridis"); axes[1].set_title("selected connected subset", fontsize=9)
axes[2].imshow(vmp, cmap="gray"); axes[2].set_title("partial valid mask", fontsize=9)
axes[3].imshow(gtp); axes[3].set_title("partial GT (outside = 0)", fontsize=9)
for a in axes: a.axis("off")
plt.tight_layout(); plt.show()
print(f"partial_frac (faces): {qm['uv_002'].get('partial_frac')} | selected faces {len(sel_faces)}/{len(F)} = {len(sel_faces)/len(F):.3f}")
print(f"texel ratio partial/full: {vmp.sum()}/{vm0.sum()} = {vmp.sum()/max(vm0.sum(),1):.3f}")
print("partial_query_layout: native (metadata semantic: connected_partial)")

## 自动一致性检查（当前样本，逐项 PASS/FAIL）

In [ ]:
import hashlib as _h
checks = []
nf = len(F)
qs = {}
for q in ["uv_000", "uv_001", "uv_002", "uv_test"]:
    qs[q] = load_file(str(d / "uv_queries" / q / "uv_address.safetensors"))
checks.append(("all queries same object (uv_faces count == mesh F)",
               all(qs[q]["uv_faces"].shape[0] == nf for q in qs)))
srcdir = Path(os.environ.get("TOPOTEX_SOURCE_ROOT", DATA / "source")) / "samples" / sid
want_tex = meta.get("source_texture_sha256")
checks.append(("source texture SHA matches",
               want_tex == _h.sha256((srcdir / "gt_texture.png").read_bytes()).hexdigest()))
checks.append(("face count matches across queries", len({qs[q]["uv_faces"].shape[0] for q in qs}) == 1))
checks.append(("face_id range valid", all(int(qs[q]["face_id"].max()) < nf for q in qs)))
ok_b = True
for q in qs:
    vm = qs[q]["valid_mask"].astype(bool)
    if vm.any():
        s = qs[q]["barycentric"].transpose(1, 2, 0)[vm].sum(-1)
        ok_b &= bool(np.abs(s - 1).max() <= 5e-3)
checks.append(("barycentric sum ~ 1", ok_b))
checks.append(("mask == (face_id >= 0)",
               all(bool(((qs[q]["face_id"] >= 0) == qs[q]["valid_mask"].astype(bool)).all()) for q in qs)))
checks.append(("no NaN/Inf", all(bool(np.isfinite(qs[q]["barycentric"]).all()) for q in qs)))
checks.append(("partial outside-mask exactly zero", bool((gtp[~vmp] == 0).all())))
need = [d / "reference.png", d / "mesh.safetensors", d / "mv.safetensors"] +        [d / "uv_queries" / q / f for q in qs for f in ("uv_address.safetensors", "gt_texture.png")]
checks.append(("all required files readable", all(p.exists() and p.stat().st_size > 0 for p in need)))
in_tr, in_te = sid in set(TRAIN), sid in set(TEST)
checks.append(("object split consistent (exactly one split)", in_tr != in_te))
fails = [n for n, ok in checks if not ok]
for n, ok in checks:
    print(("PASS " if ok else "FAIL ") + n)
assert not fails, f"consistency failures: {fails}"

## 快速人工抽检（quick: 4 train + 4 test | full: 16+16）— GT native 纹理 + split 标注

In [ ]:
n_each = 4 if MODE == "quick" else 16
rng2 = np.random.default_rng(RANDOM_SEED + 1)
pick_tr = [TRAIN[i] for i in rng2.choice(len(TRAIN), n_each, replace=False)]
pick_te = [TEST[i] for i in rng2.choice(len(TEST), n_each, replace=False)]
tiles = []
for group, tag in ((pick_tr, "train"), (pick_te, "TEST")):
    for s in group:
        gt = np.asarray(Image.open(DATASET_ROOT / "samples" / s / "uv_queries/uv_000/gt_texture.png").convert("RGB"))
        tiles.append((gt, f"{tag}:{s[:8]}"))
cols = n_each
rows_n = 2
fig, axes = plt.subplots(rows_n, cols, figsize=(2.3 * cols, 2.5 * rows_n))
axes = np.atleast_2d(axes)
for i, (im, ttl) in enumerate(tiles):
    ax = axes[i // cols][i % cols]
    ax.imshow(im); ax.set_title(ttl, fontsize=8); ax.axis("off")
plt.suptitle("random objects — native GT (top: train, bottom: unseen test)", fontsize=10)
plt.tight_layout(); plt.show()